# 10 — Multimodal evidence: tables, images, OCR, and citations

**Level:** Advanced · **Estimated time:** 90 minutes · **Scenario:** NovaTech Renewal Risk Review

NovaTech’s customer-success lead asks: **“What is the renewal value at risk in Europe, and what must happen before risk is escalated?”** The answer is split across a CSV export and a dashboard image. Build a deterministic multimodal evidence path that keeps row-level and visual-region citations.


## Learning contract

By the end, you will route a numeric question to typed table operations, retrieve an OCR region as visual evidence, keep citations bound to rows and bounding boxes, and reject weak OCR or invalid schema rather than letting an LLM guess.

This notebook is deliberately credential-free. It models the contracts around an OCR or vision provider; replace only the extraction adapter after the deterministic behavior is tested.


## 1. Multimodal RAG is not “embed everything”

Different modalities have different correctness rules:

```text
question → classify evidence needed
   ├─ numeric / filter / aggregate → typed table query → row citations
   ├─ visual label / scanned note    → OCR or vision → page + bounding-box citations
   └─ narrative policy               → text retrieval → chunk citations
                       ↓
              validate evidence and policy
                       ↓
             cited answer or abstention
```

Text similarity should not calculate currency totals. OCR text should not lose its page or region. A citation must tell a reviewer where the evidence originated and how to inspect it.


In [ ]:
import csv
from pathlib import Path
from src.rag_core.lesson_loader import load_lesson_module; globals().update({name: value for name, value in vars(load_lesson_module('curriculum/advanced/04-structured-multimodal/lab.py')).items() if not name.startswith('_')})

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path('../..')
CSV_PATH = ROOT / 'data/enterprise/multimodal/renewal-risk_q2.csv'
SVG_PATH = ROOT / 'data/enterprise/multimodal/renewal-dashboard.svg'

with CSV_PATH.open(encoding='utf-8', newline='') as handle:
    raw_rows = list(csv.DictReader(handle))

rows = [
    TableRow(
        row_id=f'renewals-q2-{index}',
        values={**row, 'renewal_value_usd': int(row['renewal_value_usd'])},
        source=f'{CSV_PATH.name}#row={index}',
    )
    for index, row in enumerate(raw_rows, start=2)
]
print(rows)


## 2. Validate the typed source before aggregating

A retrieval system can be semantically relevant and still be invalid for a calculation: a currency column could be missing, numbers could be strings, or rows could be from a different reporting period. Schema validation happens before the calculation and before answer generation.


In [ ]:
required_columns = {'customer', 'region', 'renewal_value_usd', 'risk_level', 'source_version'}
errors = validate_table_rows(rows, required_columns)
assert not errors, errors
europe_rows = filter_rows(rows, region='Europe')
summary, table_citations = aggregate_with_citations(europe_rows, 'renewal_value_usd')
print(summary)
print(table_citations)


## 3. Row-level provenance makes numbers reviewable

The calculation is deterministic: Europe is the filter, `renewal_value_usd` is the typed field, and every included row is cited. Do not cite the CSV file alone when a reviewer needs to know which rows contributed to the total.

For a SQL backend, the equivalent trace should retain the query template, bound parameters, schema/version, returned primary keys, and user/tenant policy—not a raw string assembled by the model.


In [ ]:
assert summary['sum'] == 365_000
for citation in table_citations:
    print(f'{citation.evidence_id} → {citation.source} ({citation.locator})')


## 4. OCR needs a visual locator and a confidence boundary

The dashboard contains an operator note: “Validate the Northwind account migration before escalating risk.” An OCR/vision service should return the text **and** a location. We model the extraction result with an asset ID, page, bounding box, confidence, and source path.

Open the image below in the repository to compare the evidence object with the original visual asset.


In [ ]:
# Deterministic fixture representing an OCR adapter response.
regions = [
    OcrRegion('dashboard-total', 'renewal-dashboard', 1, (80, 165, 300, 60), '$365,000', 0.99, str(SVG_PATH)),
    OcrRegion('dashboard-note', 'renewal-dashboard', 1, (80, 305, 730, 42), 'Validate the Northwind account migration before escalating risk.', 0.98, str(SVG_PATH)),
    # This is deliberately too uncertain to enter answer context.
    OcrRegion('dashboard-noise', 'renewal-dashboard', 1, (80, 305, 730, 42), 'Validate Northwind account migration', 0.46, str(SVG_PATH)),
]

note_hits = search_ocr_regions('Validate Northwind migration', regions, min_confidence=0.80)
ocr_citations = citations_for_regions(note_hits)
print(note_hits)
print(ocr_citations)


## 5. Inspect the image and the OCR region

The image is evidence, not decoration. The visual reference lets a reviewer catch an OCR mistake, a stale screenshot, or a hidden qualification that plain extracted text misses.


In [ ]:
try:
    from IPython.display import SVG, display
    display(SVG(filename=str(SVG_PATH)))
except ImportError:
    print(f'Open the image at: {SVG_PATH}')


## 6. Fuse modalities without erasing their distinctions

The final answer may combine a typed aggregate and an OCR-derived operational note, but it must keep them distinguishable. A robust response contract might contain:

- a numeric claim with table-row citations;
- a visual-note claim with page/bounding-box citations;
- an explicit recommendation labelled as a recommendation;
- the source period/version and any unresolved uncertainty.


In [ ]:
all_citations = table_citations + ocr_citations
assert citations_are_known(
    all_citations,
    row_ids={row.row_id for row in europe_rows},
    region_ids={region.region_id for region in note_hits},
)

answer = {
    'facts': [
        {'claim': f"Europe has ${summary['sum']:,} of renewal value at risk in the Q2 export.", 'citations': table_citations},
        {'claim': 'The dashboard note says to validate the Northwind account migration before escalating risk.', 'citations': ocr_citations},
    ],
    'recommendation': 'Validate the migration with the account owner, then escalate only if the evidence remains consistent.',
}
answer


## 7. Deliberate failures

### Failure A — text retrieval for a calculation

If a model is handed the CSV as prose, it might miss a row, confuse the reporting period, or add values that were not filtered. Use typed filters and aggregation instead.

### Failure B — low-confidence OCR in context

If OCR confidence is low, the system should retrieve the original image for human review, run a second extraction, or abstain. It should not treat an uncertain region as a fact.

### Failure C — citation without a locator

“dashboard.svg” is not enough for a reviewer to find a visual claim. Preserve the page and bounding box, or an equivalent layout/region reference.


In [ ]:
# The low-confidence extraction is intentionally excluded at the default threshold.
assert 'dashboard-noise' not in {region.region_id for region in note_hits}

# Simulate schema drift: a new export omitted the currency amount.
broken_rows = [TableRow('broken-1', {'customer': 'Northwind', 'region': 'Europe'}, 'broken.csv#row=2')]
print(validate_table_rows(broken_rows, required_columns))


## 8. Experiment: policy choices

Change exactly one variable per run:

1. Raise `min_confidence` from `0.80` to `0.995`. Which claim can no longer be made?
2. Add a `tenant` or `allowed_roles` value to each row and enforce it before filtering. Which citations disappear for an unauthorized user?
3. Add a second dashboard screenshot with a different period. What version/freshness rule prevents mixing Q1 notes with Q2 rows?

**Success criterion:** each rejected or included item has an explicit policy reason that can be shown in a trace.


In [ ]:
for threshold in (0.80, 0.95, 0.995):
    accepted = search_ocr_regions('Validate Northwind migration', regions, min_confidence=threshold)
    print(threshold, [region.region_id for region in accepted])


## 9. Production design checklist

- Parse each source with a modality-appropriate extractor; preserve tables, pages, layout, units, and versions.
- Enforce document, row, and asset authorization before any content reaches a model context.
- Use typed tools/SQL for calculations and schema validation for dates, currencies, and units.
- Store OCR confidence and visual locators; route low-confidence evidence to review or abstention.
- Keep claims, evidence IDs, modality, locators, and trace IDs together until rendering.
- Evaluate table accuracy, OCR transcription, citation correctness/completeness, latency, and cross-tenant leakage separately.


## Checkpoint

1. Why is semantic text retrieval not an acceptable calculation engine for a currency total?
2. Which fields make an OCR citation reviewable?
3. What should happen when only a low-confidence OCR region matches a high-risk question?
4. How would you prevent a user from retrieving another tenant’s CSV rows or dashboard annotations?

### References

- [Structured and multimodal RAG module](../../curriculum/advanced/04-structured-multimodal/README.md)
- [Document AI overview — Google Cloud](https://cloud.google.com/document-ai/docs/overview) (vendor documentation)
- [Azure AI Document Intelligence overview](https://learn.microsoft.com/azure/ai-services/document-intelligence/overview) (vendor documentation)
- [RAG evaluation guide](../../docs/evaluation.md)
